In [9]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FastEmbedEmbeddings

In [10]:
DATA_DIR = "pdfs"
VECTORSTORE_PATH = "faiss_index"

In [11]:
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)
    print(f"Directory '{DATA_DIR}' created. Please, add your PDFs there.")
    exit()

In [12]:
def create_vectordb():
    print(f"Loading PDFs from directory: {DATA_DIR}")
    try:
        pdf_loader = PyPDFDirectoryLoader(DATA_DIR, recursive = True)
        documents = pdf_loader.load()
        
        if not documents:
            print(f"No PDF documents found in '{DATA_DIR}'. Please check the directory.")
            return False
            
        print(f"Loaded {len(documents)} PDF pages/documents.")

    except Exception as e:
        print(f"Error loading PDFs: {e}")
        return False

    print("Dividing documents into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 150)
    
    docs_split = text_splitter.split_documents(documents)
    print(f"Documents split into {len(docs_split)} chunks.")

    print("Initializing embedding model (FastEmbed)...")
    embedding_model = FastEmbedEmbeddings(model_name = "BAAI/bge-base-en")

    print("Creating vector index FAISS...")
    try:
        vector_store = FAISS.from_documents(docs_split, embedding_model)
        print("FAISS index created in memory.")
        vector_store.save_local(VECTORSTORE_PATH)
        print(f"FAISS index saved locally in: {VECTORSTORE_PATH}")
        return True
    except Exception as e:
        print(f"Error creating or saving index FAISS: {e}")
        return False

In [13]:
if __name__ == "__main__":
    print("\nStarting RAG configuration process...")

    if create_vectordb():
        print("RAG configuration completed successfully!")
        print(f"The vector index is saved in '{VECTORSTORE_PATH}'.")
        print(f"Make sure your PDFs are in the folder '{DATA_DIR}'.\n")

    else:
        print("\nRAG configuration failed. Check the errors above.")


Starting RAG configuration process...
Loading PDFs from directory: pdfs
Loaded 2 PDF pages/documents.
Dividing documents into chunks...
Documents split into 4 chunks.
Initializing embedding model (FastEmbed)...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\krupc\AppData\Local\Temp\fastembed_cache\models--Qdrant--fast-bge-base-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 5 files: 100%|██████████| 5/5 [

Creating vector index FAISS...
FAISS index created in memory.
FAISS index saved locally in: faiss_index
RAG configuration completed successfully!
The vector index is saved in 'faiss_index'.
Make sure your PDFs are in the folder 'pdfs'.

